In [ ]:
from mpramnist.Ernst2016 import ErnstDataset
from mpramnist.Ernst2016 import LitModel_Ernst

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

from mpramnist import transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1096
NUM_WORKERS = 8

print(ErnstDataset.CELL_TYPES)

In [ ]:
# preprocessing
train_transform = t.Compose([t.ReverseComplement(0.5),t.Seq2Tensor(),])
test_transform = t.Compose([t.Seq2Tensor(),])
# load the data
cell_type = [
    "k562_minp_avg",
    "k562_sv40p_avg",
    "hepg2_minp_avg",
    "hepg2_sv40p_avg",
]
train_dataset = ErnstDataset(split="train",cell_type=cell_type,transform=train_transform,root="../data/",)  # for needed folds
val_dataset = ErnstDataset(split="val",cell_type=cell_type,transform=test_transform,root="../data/",)  # use "val" for default validation set
test_dataset = ErnstDataset(split="test",cell_type=cell_type,transform=test_transform,root="../data/",)  # use "test" for default test set

print(len(train_dataset),len(val_dataset),len(test_dataset))

# encapsulate data into dataloader form
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = len(cell_type)

100%|██████████| 163M/163M [04:24<00:00, 616kB/s]  


val: After filtering duplicates: 19833 sequences in val
test: After filtering duplicates: 10130 sequences in test


In [ ]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Ernst(model=model,loss=nn.MSELoss(),cell_types=cell_type,weight_decay=1e-1,lr=1e-2,print_each=1)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=False,
    num_sanity_val_steps=0,
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

In [ ]:
def meaned_prediction(forw, rev, trainer, seq_model, name, num):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef(num_outputs=num)
    print(name + " Pearson correlation")

    return pears(mean_forw, targets)

forw_transform = t.Compose([t.Seq2Tensor()])
rev_transform = t.Compose([t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = ErnstDataset(split="test",cell_type=cell_type,transform=forw_transform,root="../data/",)
test_rev = ErnstDataset(split="test",cell_type=cell_type,transform=rev_transform,root="../data/",)

forw = DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw, rev, trainer, seq_model, "Sharpr", len(cell_type))

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Sharpr Pearson correlation


tensor([0.4016, 0.2175, 0.3214, 0.3212])